# Retrieval Evaluation

Evaluate top-3 evidence retrieval against `FinalDataset/claims_merged.csv`.

Experiment matrix:

- refined model: Gemini 2.5 Flash, GPT-4o mini
- Qdrant collection: `fixed_size`, `semantic`
- image vector: `image_vector`, `image_vector_finetuned`
- reranker: off, on

Outputs are saved under `database/retrieval_eval_outputs/`.

In [8]:
from __future__ import annotations

import hashlib
import json
import math
import re
import unicodedata
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseVector
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"database", "refined"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

QDRANT_URL = "http://localhost:6333"
GROUND_TRUTH_CSV = PROJECT_ROOT / "FinalDataset" / "claims_merged.csv"
REFINED_FILES = {
    "gemini-2.5-flash": PROJECT_ROOT / "refined" / "refined_outputs_openrouter" / "refined_gemini-2.5-flash.csv",
    "gpt4o_mini": PROJECT_ROOT / "refined" / "refined_outputs_openrouter" / "refined_gpt4o_mini.csv",
}
OUTPUT_DIR = PROJECT_ROOT / "database" / "retrieval_eval_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLLECTIONS = ["fixed_size", "semantic"]
IMAGE_VECTOR_VARIANTS = {
    "clip": "image_vector",
    "clip_finetuned": "image_vector_finetuned",
}
RUN_RERANKER_VALUES = [False, True]  # First run baseline. Change to [True] or [False, True] for reranker experiments.
ROW_LIMIT = None # Set to a small number such as 20 for smoke testing.

TEXT_VECTOR = "text_vector"
SPARSE_VECTOR = "sparse"
BKVEC_MODEL = "bkai-foundation-models/vietnamese-bi-encoder"
IMG_MODEL = "sentence-transformers/clip-ViT-B-32"
IMG_MODEL_FINETUNED = PROJECT_ROOT / "models" / "clip-vit-b32-finetuned-final-final" / "best"
CROSS_ENCODER_MODEL = "namdp-ptit/ViRanker"
CROSS_ENCODER_MAX_LENGTH = 512

CANDIDATES_PER_QUERY = 10
MAX_TEXT_QUERIES = 6
MAX_VISUAL_QUERIES = 6
TOP_K_VALUES = [3, 10, 20]
FINAL_TOP_K = 3
RRF_K = 60
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

client = QdrantClient(url=QDRANT_URL)
print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Qdrant collections:", [c.name for c in client.get_collections().collections])


Project root: d:\FactCheckPipeline
Output dir: d:\FactCheckPipeline\database\retrieval_eval_outputs
Qdrant collections: ['fixed_size', 'semantic']


## Load Data

In [9]:
gt = pd.read_csv(GROUND_TRUTH_CSV)
refined_by_model = {}
for alias, path in REFINED_FILES.items():
    frame = pd.read_csv(path)
    if "refine_error" in frame.columns:
        frame = frame[frame["refine_error"].fillna("").astype(str).str.strip().eq("")].copy()
    refined_by_model[alias] = frame
    print(alias, frame.shape)

print("ground_truth", gt.shape)
display(gt.head(3))


gemini-2.5-flash (1293, 25)
gpt4o_mini (1293, 25)
ground_truth (1293, 9)


,id,claim,image,text_evidences,text_evidences_url,image_evidences,image_evidence_path,reason,label
0,1,Công an tỉnh Thái Bình đã khởi tố 10 đối tượng...,media/post_1_cmt_img_0.jpg,Công an tỉnh Thái Bình vừa triệt phá đường dây...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh bên phải hiển thị một nhóm khoảng 10-...,media/post_1_cmt_img_0.jpg,Claim này được đánh giá là supported vì văn bả...,supported
1,2,Các đối tượng cầm đầu đường dây lừa đảo đã thu...,media/post_1_cmt_img_0.jpg,Chúng thuê nhà tại Hà Nội và TP. Hồ Chí Minh l...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh bên trái hiển thị nhiều điện thoại di...,media/post_1_cmt_img_0.jpg,Claim này được đánh giá là supported vì văn bả...,supported
2,3,Thượng úy Nguyễn Đức Phước là Điều tra viên th...,media/post_1_cmt_img_0.jpg,Đại úy Nguyễn Đức Phước - Phó Đội trưởng Đội Đ...,https://www.facebook.com/mps.gov/posts/pfbid02...,Hình ảnh không cung cấp thông tin về vai trò c...,media/post_1_cmt_img_0.jpg,Claim này bị đánh giá là refuted vì văn bản cu...,refuted


## Matching Rules

Text evidence hit priority:

1. exact URL match
2. token coverage against each gold evidence fragment
3. token F1 against each gold evidence fragment

The coverage rule is important because retrieved chunks are often much longer than the gold evidence sentence. Pure F1 would unfairly penalize long but correct chunks.

In [10]:
def normalize_text(text: Any) -> str:
    text = str(text or "").lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text

def tokens(text: Any) -> list[str]:
    return TOKEN_RE.findall(normalize_text(text))

def token_scores(retrieved_text: Any, gold_text: Any) -> dict[str, float]:
    r = set(tokens(retrieved_text))
    g = set(tokens(gold_text))
    if not r or not g:
        return {"precision": 0.0, "coverage": 0.0, "f1": 0.0}
    inter = len(r & g)
    precision = inter / len(r)
    coverage = inter / len(g)
    f1 = 2 * precision * coverage / (precision + coverage) if precision + coverage else 0.0
    return {"precision": precision, "coverage": coverage, "f1": f1}

def split_gold_text_evidences(value: Any) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    return [part.strip() for part in str(value).split(" | ") if part.strip()]

def normalize_url(value: Any) -> str:
    text = str(value or "").strip().lower()
    return text.rstrip("/")

def normalize_path(value: Any) -> str:
    text = str(value or "").replace("\\", "/").strip().lower()
    while text.startswith("../"):
        text = text[3:]
    if text.startswith("finaldataset/"):
        text = text[len("finaldataset/"):]
    return text

def text_evidence_match(payload: dict[str, Any], gold_row: pd.Series) -> dict[str, Any]:
    retrieved_url = normalize_url(payload.get("url", ""))
    gold_url = normalize_url(gold_row.get("text_evidences_url", ""))
    if retrieved_url and gold_url and retrieved_url == gold_url:
        return {"hit": True, "method": "url", "coverage": 1.0, "f1": 1.0}

    retrieved_text = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "url"])
    best = {"hit": False, "method": "none", "coverage": 0.0, "f1": 0.0}
    for gold_text in split_gold_text_evidences(gold_row.get("text_evidences", "")):
        scores = token_scores(retrieved_text, gold_text)
        if scores["coverage"] > best["coverage"] or scores["f1"] > best["f1"]:
            best = {"hit": False, "method": "text_overlap", **scores}
    best["hit"] = best["coverage"] >= 0.60 or best["f1"] >= 0.45
    return best

def image_evidence_match(payload: dict[str, Any], gold_row: pd.Series) -> dict[str, Any]:
    retrieved = normalize_path(payload.get("image_path", ""))
    gold = normalize_path(gold_row.get("image_evidence_path", ""))
    hit = bool(retrieved and gold and retrieved == gold)
    return {"hit": hit, "method": "path" if hit else "none"}


## Query Pack And Embeddings

In [11]:
def safe_json(value: Any, default: Any):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    try:
        return json.loads(str(value))
    except Exception:
        return default

def dedupe(items: list[Any], max_items: int | None = None) -> list[str]:
    out, seen = [], set()
    for item in items:
        text = str(item or "").strip()
        key = normalize_text(text)
        if text and key not in seen:
            seen.add(key)
            out.append(text)
    return out[:max_items] if max_items else out

def build_query_pack(row: pd.Series) -> dict[str, Any]:
    search_queries = safe_json(row.get("refined_search_queries"), {})
    claim_atoms = safe_json(row.get("refined_claim_atoms"), [])
    visual_observations = safe_json(row.get("refined_visual_observations"), [])
    retrieval_focus = safe_json(row.get("refined_retrieval_focus"), {})
    verification_targets = safe_json(row.get("refined_verification_targets"), [])

    atom_queries = []
    for atom in claim_atoms if isinstance(claim_atoms, list) else []:
        if isinstance(atom, dict):
            atom_queries.extend(atom.get("retrieval_queries", []))

    visual_terms = []
    for obs in visual_observations if isinstance(visual_observations, list) else []:
        if isinstance(obs, dict):
            visual_terms.append(obs.get("text", ""))
            visual_terms.extend(obs.get("visible_evidence", []))

    text_queries = dedupe([
        row.get("refined_primary_retrieval_query", ""),
        row.get("refined_normalized_claim", ""),
        *search_queries.get("semantic", []),
        *atom_queries,
        *verification_targets,
    ], MAX_TEXT_QUERIES)

    keyword_query = " ".join(dedupe([*search_queries.get("keywords", []), *verification_targets]))
    visual_queries = dedupe([*search_queries.get("visual", []), *visual_terms], MAX_VISUAL_QUERIES)
    if retrieval_focus.get("cross_modal", False):
        visual_queries = dedupe([*visual_queries, row.get("refined_primary_retrieval_query", "")], MAX_VISUAL_QUERIES)
    return {"text_queries": text_queries, "keyword_query": keyword_query, "visual_queries": visual_queries, "retrieval_focus": retrieval_focus}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device)
bkai_model = SentenceTransformer(BKVEC_MODEL, device=device)
clip_model = SentenceTransformer(IMG_MODEL, device=device)
clip_finetuned_model = SentenceTransformer(str(IMG_MODEL_FINETUNED), device=device)

def embed_text_bkai(texts: list[str]) -> list[list[float]]:
    return bkai_model.encode(texts, batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).tolist()

def embed_text_clip(texts: list[str], image_variant: str) -> list[list[float]]:
    model = clip_finetuned_model if image_variant == "clip_finetuned" else clip_model
    return model.encode(texts, batch_size=16, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).tolist()

def sparse_vector(text: str) -> SparseVector:
    counts = {}
    for token in TOKEN_RE.findall(str(text or "").lower()):
        digest = hashlib.blake2b(token.encode("utf-8"), digest_size=8).digest()
        idx = int.from_bytes(digest, "big") % 2_147_483_647
        counts[idx] = counts.get(idx, 0) + 1
    indices = sorted(counts)
    values = [1.0 + math.log(counts[idx]) for idx in indices]
    norm = math.sqrt(sum(v * v for v in values)) or 1.0
    return SparseVector(indices=indices, values=[v / norm for v in values])


device cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Retrieval, Fusion, Optional Reranker

In [12]:
def modality_filter(modality: str) -> models.Filter:
    return models.Filter(must=[models.FieldCondition(key="modality", match=models.MatchValue(value=modality))])

def query_qdrant(collection: str, query: Any, vector_name: str, modality: str, limit: int):
    return client.query_points(
        collection_name=collection,
        query=query,
        using=vector_name,
        query_filter=modality_filter(modality),
        limit=limit,
        with_payload=True,
        with_vectors=False,
    ).points

def generate_candidates(row: pd.Series, collection: str, image_variant: str) -> dict[str, list[dict[str, Any]]]:
    pack = build_query_pack(row)
    branches = {}

    text_hits = []
    if pack["text_queries"]:
        for q, v in zip(pack["text_queries"], embed_text_bkai(pack["text_queries"])):
            text_hits.extend({"point": p, "query": q} for p in query_qdrant(collection, v, TEXT_VECTOR, "text", CANDIDATES_PER_QUERY))
    branches["text_dense"] = text_hits

    sparse_hits = []
    if pack["keyword_query"].strip():
        sparse_hits = [{"point": p, "query": pack["keyword_query"]} for p in query_qdrant(collection, sparse_vector(pack["keyword_query"]), SPARSE_VECTOR, "text", CANDIDATES_PER_QUERY * 2)]
    branches["text_sparse"] = sparse_hits

    image_hits = []
    vector_name = IMAGE_VECTOR_VARIANTS[image_variant]
    if pack["visual_queries"]:
        for q, v in zip(pack["visual_queries"], embed_text_clip(pack["visual_queries"], image_variant)):
            image_hits.extend({"point": p, "query": q} for p in query_qdrant(collection, v, vector_name, "image", CANDIDATES_PER_QUERY))
    branches[f"image_{image_variant}"] = image_hits
    return branches

# Keep image branch weights equal so original CLIP vs fine-tuned CLIP is a fair A/B test.
# Each experiment uses exactly one image vector variant, selected by IMAGE_VECTOR_VARIANTS.
BRANCH_WEIGHTS = {"text_dense": 1.00, "text_sparse": 1.00, "image_clip": 1.00, "image_clip_finetuned": 1.00}

def weighted_rrf(branches: dict[str, list[dict[str, Any]]]) -> list[dict[str, Any]]:
    fused = {}
    for branch, records in branches.items():
        seen = set()
        for rank, record in enumerate(records, start=1):
            p = record["point"]
            key = str(p.id)
            if key in seen:
                continue
            seen.add(key)
            item = fused.setdefault(key, {"point_id": key, "payload": p.payload or {}, "rrf_score": 0.0, "branches": [], "best_qdrant_score": float(p.score)})
            item["rrf_score"] += BRANCH_WEIGHTS.get(branch, 1.0) / (RRF_K + rank)
            item["best_qdrant_score"] = max(item["best_qdrant_score"], float(p.score))
            item["branches"].append({"branch": branch, "rank": rank, "score": float(p.score), "query": record.get("query", "")})
    return sorted(fused.values(), key=lambda x: x["rrf_score"], reverse=True)

cross_encoder = None
def get_cross_encoder():
    global cross_encoder
    if cross_encoder is None:
        from sentence_transformers import CrossEncoder
        cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, device=device, max_length=CROSS_ENCODER_MAX_LENGTH)
    return cross_encoder

def rerank_items(items: list[dict[str, Any]], row: pd.Series, use_reranker: bool) -> list[dict[str, Any]]:
    query = str(row.get("refined_primary_retrieval_query") or row.get("claim") or "")
    ce = get_cross_encoder() if use_reranker else None
    for item in items:
        payload = item["payload"]
        branch_bonus = min(len({b["branch"] for b in item["branches"]}) * 0.025, 0.10)
        reranker_boost = 0.0
        if ce is not None and payload.get("modality") == "text":
            passage = " ".join(str(payload.get(k, "")) for k in ["title", "text", "source", "date"])
            if passage.strip():
                raw = float(ce.predict([(query, passage[:3000])])[0])
                reranker_boost = (1.0 / (1.0 + math.exp(-raw))) * 0.25
        item["reranker_boost"] = reranker_boost
        item["final_score"] = item["rrf_score"] + branch_bonus + reranker_boost
    return sorted(items, key=lambda x: x["final_score"], reverse=True)

def retrieve(row: pd.Series, collection: str, image_variant: str, use_reranker: bool) -> list[dict[str, Any]]:
    return rerank_items(weighted_rrf(generate_candidates(row, collection, image_variant)), row, use_reranker)


## Metrics

In [13]:
def evaluate_ranked_items(ranked: list[dict[str, Any]], gold_row: pd.Series) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    rows = []
    for rank, item in enumerate(ranked[:max(TOP_K_VALUES)], start=1):
        payload = item["payload"]
        text_match = text_evidence_match(payload, gold_row) if payload.get("modality") == "text" else {"hit": False, "method": "none", "coverage": 0.0, "f1": 0.0}
        image_match = image_evidence_match(payload, gold_row) if payload.get("modality") == "image" else {"hit": False, "method": "none"}
        evidence_hit = bool(text_match["hit"] or image_match["hit"])
        rows.append({
            "rank": rank,
            "point_id": item["point_id"],
            "modality": payload.get("modality", ""),
            "final_score": item.get("final_score", 0.0),
            "rrf_score": item.get("rrf_score", 0.0),
            "reranker_boost": item.get("reranker_boost", 0.0),
            "best_qdrant_score": item.get("best_qdrant_score", 0.0),
            "branches": json.dumps(item.get("branches", []), ensure_ascii=False),
            "hit": evidence_hit,
            "text_hit": bool(text_match["hit"]),
            "image_hit": bool(image_match["hit"]),
            "text_match_method": text_match.get("method", "none"),
            "text_coverage": text_match.get("coverage", 0.0),
            "text_f1": text_match.get("f1", 0.0),
            "title": payload.get("title", ""),
            "url": payload.get("url", ""),
            "image_path": payload.get("image_path", ""),
            "text": str(payload.get("text", ""))[:1000],
        })

    metrics = {}
    for k in TOP_K_VALUES:
        top = rows[:k]
        metrics[f"evidence_recall_at_{k}"] = int(any(r["hit"] for r in top))
        metrics[f"text_recall_at_{k}"] = int(any(r["text_hit"] for r in top))
        metrics[f"image_recall_at_{k}"] = int(any(r["image_hit"] for r in top))
        metrics[f"full_recall_at_{k}"] = int(any(r["text_hit"] for r in top) and any(r["image_hit"] for r in top))
    first_hit_rank = next((r["rank"] for r in rows if r["hit"]), None)
    metrics["mrr_at_3"] = 1 / first_hit_rank if first_hit_rank and first_hit_rank <= 3 else 0.0
    metrics["first_hit_rank"] = first_hit_rank or 0
    return rows, metrics

def summarize_metrics(per_claim_metrics: list[dict[str, Any]]) -> dict[str, Any]:
    frame = pd.DataFrame(per_claim_metrics)
    metric_cols = [c for c in frame.columns if c.endswith(tuple(str(k) for k in TOP_K_VALUES)) or c in {"mrr_at_3"}]
    return {col: float(frame[col].mean()) for col in metric_cols}


## Run Experiments

In [14]:
all_detail_rows = []
all_claim_metric_rows = []
summary_rows = []

for refiner_alias, refined in refined_by_model.items():
    eval_frame = refined.merge(gt, on=["id", "claim", "image"], how="inner", suffixes=("", "_gold"))
    if ROW_LIMIT is not None:
        eval_frame = eval_frame.head(ROW_LIMIT).copy()
    print(f"\n{refiner_alias}: evaluating {len(eval_frame)} matched rows")
    for collection in COLLECTIONS:
        for image_variant in IMAGE_VECTOR_VARIANTS:
            for use_reranker in RUN_RERANKER_VALUES:
                experiment_id = f"{refiner_alias}__{collection}__{image_variant}__reranker_{int(use_reranker)}"
                print("Running", experiment_id)
                detail_rows = []
                claim_metric_rows = []
                for _, row in tqdm(eval_frame.iterrows(), total=len(eval_frame), desc=experiment_id):
                    ranked = retrieve(row, collection, image_variant, use_reranker)
                    evidence_rows, metrics = evaluate_ranked_items(ranked, row)
                    base = {
                        "experiment_id": experiment_id,
                        "refiner": refiner_alias,
                        "collection": collection,
                        "image_variant": image_variant,
                        "use_reranker": use_reranker,
                        "id": row["id"],
                        "claim": row["claim"],
                        "label": row.get("label", ""),
                        "gold_text_url": row.get("text_evidences_url", ""),
                        "gold_image_path": row.get("image_evidence_path", ""),
                    }
                    for ev in evidence_rows:
                        detail_rows.append({**base, **ev})
                    claim_metric_rows.append({**base, **metrics})

                detail_df = pd.DataFrame(detail_rows)
                claim_metrics_df = pd.DataFrame(claim_metric_rows)
                summary = summarize_metrics(claim_metric_rows)
                summary_rows.append({
                    "experiment_id": experiment_id,
                    "refiner": refiner_alias,
                    "collection": collection,
                    "image_variant": image_variant,
                    "use_reranker": use_reranker,
                    "num_claims": len(claim_metric_rows),
                    **summary,
                })

                detail_path = OUTPUT_DIR / f"retrieval_details_{experiment_id}.csv"
                claim_metrics_path = OUTPUT_DIR / f"claim_metrics_{experiment_id}.csv"
                detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
                claim_metrics_df.to_csv(claim_metrics_path, index=False, encoding="utf-8-sig")
                all_detail_rows.extend(detail_rows)
                all_claim_metric_rows.extend(claim_metric_rows)

summary_df = pd.DataFrame(summary_rows).sort_values(["evidence_recall_at_3", "mrr_at_3"], ascending=False)
all_details_df = pd.DataFrame(all_detail_rows)
all_claim_metrics_df = pd.DataFrame(all_claim_metric_rows)

summary_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False, encoding="utf-8-sig")
all_details_df.to_csv(OUTPUT_DIR / "retrieval_results_long.csv", index=False, encoding="utf-8-sig")
all_claim_metrics_df.to_csv(OUTPUT_DIR / "claim_metrics_long.csv", index=False, encoding="utf-8-sig")

display(summary_df)
print("Saved outputs to", OUTPUT_DIR)



gemini-2.5-flash: evaluating 1293 matched rows
Running gemini-2.5-flash__fixed_size__clip__reranker_0


gemini-2.5-flash__fixed_size__clip__reranker_0:   0%|          | 0/1293 [00:00<?, ?it/s]

Running gemini-2.5-flash__fixed_size__clip__reranker_1


gemini-2.5-flash__fixed_size__clip__reranker_1:   0%|          | 0/1293 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

gpt4o_mini__semantic__clip_finetuned__reranker_0:   0%|          | 0/1293 [00:00<?, ?it/s]

Running gpt4o_mini__semantic__clip_finetuned__reranker_1


gpt4o_mini__semantic__clip_finetuned__reranker_1:   0%|          | 0/1293 [00:00<?, ?it/s]

,experiment_id,refiner,collection,image_variant,use_reranker,num_claims,evidence_recall_at_3,text_recall_at_3,image_recall_at_3,full_recall_at_3,evidence_recall_at_10,text_recall_at_10,image_recall_at_10,full_recall_at_10,evidence_recall_at_20,text_recall_at_20,image_recall_at_20,full_recall_at_20,mrr_at_3
13,gpt4o_mini__semantic__clip__reranker_1,gpt4o_mini,semantic,clip,True,1293,0.921887,0.921887,0.000000,0.000000,0.953596,0.953596,0.000000,0.000000,0.962104,0.962104,0.000000,0.000000,0.878577
15,gpt4o_mini__semantic__clip_finetuned__reranker_1,gpt4o_mini,semantic,clip_finetuned,True,1293,0.921887,0.921887,0.000000,0.000000,0.953596,0.953596,0.000000,0.000000,0.962104,0.962104,0.000000,0.000000,0.878577
5,gemini-2.5-flash__semantic__clip__reranker_1,gemini-2.5-flash,semantic,clip,True,1293,0.921114,0.921114,0.000000,0.000000,0.955916,0.955916,0.000000,0.000000,0.963650,0.963650,0.000000,0.000000,0.882315
7,gemini-2.5-flash__semantic__clip_finetuned__re...,gemini-2.5-flash,semantic,clip_finetuned,True,1293,0.921114,0.921114,0.000000,0.000000,0.955916,0.955916,0.000000,0.000000,0.963650,0.963650,0.000000,0.000000,0.882315
1,gemini-2.5-flash__fixed_size__clip__reranker_1,gemini-2.5-flash,fixed_size,clip,True,1293,0.895592,0.895592,0.000000,0.000000,0.941222,0.941222,0.000000,0.000000,0.951276,0.951276,0.000000,0.000000,0.854988
3,gemini-2.5-flash__fixed_size__clip_finetuned__...,gemini-2.5-flash,fixed_size,clip_finetuned,True,1293,0.895592,0.895592,0.000000,0.000000,0.941222,0.941222,0.000000,0.000000,0.951276,0.951276,0.000000,0.000000,0.854988
9,gpt4o_mini__fixed_size__clip__reranker_1,gpt4o_mini,fixed_size,clip,True,1293,0.890178,0.890178,0.000000,0.000000,0.935808,0.935808,0.000000,0.000000,0.948183,0.948183,0.000000,0.000000,0.850606
11,gpt4o_mini__fixed_size__clip_finetuned__rerank...,gpt4o_mini,fixed_size,clip_finetuned,True,1293,0.890178,0.890178,0.000000,0.000000,0.935808,0.935808,0.000000,0.000000,0.948183,0.948183,0.000000,0.000000,0.850606
4,gemini-2.5-flash__semantic__clip__reranker_0,gemini-2.5-flash,semantic,clip,False,1293,0.885538,0.885538,0.001547,0.001547,0.936582,0.936582,0.010828,0.010828,0.958237,0.958237,0.015468,0.015468,0.824181
6,gemini-2.5-flash__semantic__clip_finetuned__re...,gemini-2.5-flash,semantic,clip_finetuned,False,1293,0.885538,0.885538,0.000773,0.000773,0.936582,0.936582,0.004640,0.004640,0.958237,0.958237,0.006961,0.006961,0.824181


Saved outputs to d:\FactCheckPipeline\database\retrieval_eval_outputs
